# B2-Li 760 r8: ещё 3 эпохи с hard-pixel loss

Старт: EMA после третьей эпохи `disentangle_b2_li760_r8_all_data_ft`,
сохранённая в `runs/disentangle_b2_li760_r8_all_data_ft_ep3/ckpt/last.pt`.
Новый optimizer и EMA; состояние optimizer исходного запуска не переносится.

- Ещё 3 полных прохода по всем размеченным данным: train + development + holdout и проверенные оригиналы.
- Постоянный LR с первого шага: encoder `1e-5`, JPEG/head `3e-5`. Scheduler и LR warmup отключены.
- Исходный loss + `0.1 * (hard_positive_BCE + hard_negative_BCE)`; коэффициент фиксирован.
- По каждому изображению отдельно выбираются худшие 10% допустимых пикселей внутри GT и вне GT.
- Дополнительный loss исключает полосу радиусом 2 пикселя вокруг границы на сетке target; обычный loss использует всю маску.
- Если допустимая область пуста, её вклад равен нулю. Для чистых изображений действует только hard-negative компонент.
- Полные кадры и аугментации унаследованы от исходного дотюна. Архитектура и inference не меняются.

Валидации, подбора порогов и независимого holdout нет. Test не используется.
Этот запуск не является контролируемой оценкой прироста качества: одновременно меняются loss и LR schedule.
Loss-компоненты записываются отдельно в логи. Результат: новый run / `ckpt/last.pt` (model и ema).
Последняя ячейка запускает обучение; повторный запуск продолжает checkpoint нового run.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_r8_all_data_hard_pixel_ft'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
source_checkpoint = cfg.paths.runs_path / cfg.train.finetune_from
resume_checkpoint = cfg.paths.runs_path / cfg.run_name / 'ckpt' / 'last.pt'
if not resume_checkpoint.is_file() and not source_checkpoint.is_file():
    raise FileNotFoundError(f'Missing epoch-3 checkpoint: {source_checkpoint}')
print('Source:', source_checkpoint)
print('Resume checkpoint:', resume_checkpoint if resume_checkpoint.is_file() else 'new run')
print('Scheduler:', cfg.train.scheduler)
print('Constant LR:', cfg.train.encoder_lr, cfg.train.jpeg_lr, cfg.train.head_lr)
print('Hard-pixel loss:', cfg.loss.hard_pixel_weight, cfg.loss.hard_pixel_fraction, cfg.loss.hard_pixel_radius)
cfg


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
